# Самостоятельная работа (Spark MLlib)

In [1]:
import os, sys, warnings
warnings.filterwarnings("ignore")
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import *
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, VectorAssembler, StandardScaler
from pyspark.ml.regression import LinearRegression
from pyspark.ml.classification import LogisticRegression, RandomForestClassifier, GBTClassifier
from pyspark.ml.evaluation import RegressionEvaluator, BinaryClassificationEvaluator
from pyspark.ml.tuning import ParamGridBuilder, CrossValidator

spark = SparkSession.builder.master("local[*]").appName("Spark_MLlib").getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

## Задание №1: Векторизация и масштабирование признаков

In [2]:
data = [
    (1, "Toyota", 2015, 12000, 1.6),
    (2, "Ford", 2018, 45000, 2.0),
    (3, "Toyota", 2020, 80000, 2.5),
    (4, "Nissan", 2012, 8000, 1.5),
    (5, "Ford", 2019, 50000, 2.0),
    (6, "BMW", 2021, 150000, 3.0),
    (7, "Nissan", 2016, 20000, 1.6)
]
columns = ["id", "brand", "year", "price", "engine_vol"]
df_cars = spark.createDataFrame(data, columns)

indexer = StringIndexer(inputCol="brand", outputCol="brand_idx")
df_idx = indexer.fit(df_cars).transform(df_cars)

assembler = VectorAssembler(inputCols=["brand_idx", "year", "engine_vol"], outputCol="raw_features")
df_vec = assembler.transform(df_idx)

scaler = StandardScaler(inputCol="raw_features", outputCol="scaled_features", withStd=True, withMean=True)
df_scaled = scaler.fit(df_vec).transform(df_vec)

df_scaled.select("id", "brand", "brand_idx", "raw_features").show(truncate=False)
df_scaled.select("id", "price", "scaled_features").show(truncate=False)

+---+------+---------+----------------+
|id |brand |brand_idx|raw_features    |
+---+------+---------+----------------+
|1  |Toyota|2.0      |[2.0,2015.0,1.6]|
|2  |Ford  |0.0      |[0.0,2018.0,2.0]|
|3  |Toyota|2.0      |[2.0,2020.0,2.5]|
|4  |Nissan|1.0      |[1.0,2012.0,1.5]|
|5  |Ford  |0.0      |[0.0,2019.0,2.0]|
|6  |BMW   |3.0      |[3.0,2021.0,3.0]|
|7  |Nissan|1.0      |[1.0,2016.0,1.6]|
+---+------+---------+----------------+



+---+------+-------------------------------------------------------------+
|id |price |scaled_features                                              |
+---+------+-------------------------------------------------------------+
|1  |12000 |[0.6419407387663693,-0.7262730392025365,-0.7793741540434868] |
|2  |45000 |[-1.155493329779465,0.22696032475081973,-0.05195827693623229]|
|3  |80000 |[0.6419407387663693,0.8624492340530572,0.8573115694478359]   |
|4  |8000  |[-0.2567762955065478,-1.6795064031558926,-0.9612281233203006]|
|5  |50000 |[-1.155493329779465,0.5447047794019384,-0.05195827693623229] |
|6  |150000|[1.5406577730392863,1.180193688704176,1.7665814158319042]    |
|7  |20000 |[-0.2567762955065478,-0.4085285845514177,-0.7793741540434868]|
+---+------+-------------------------------------------------------------+



## Задание №2: Прогнозирование тренда (линейная регрессия)

In [3]:
data = [
    (1, 2010, 150000, 1.6, 5000),
    (2, 2015, 80000, 2.0, 12000),
    (3, 2018, 40000, 1.5, 18000),
    (4, 2020, 15000, 2.5, 30000),
    (5, 2012, 120000, 1.6, 7500),
    (6, 2022, 5000, 3.0, 50000),
    (7, 2016, 60000, 2.0, 15000),
    (8, 2019, 30000, 2.5, 25000)
]
columns = ["id", "year", "mileage", "engine_vol", "price"]
df_reg = spark.createDataFrame(data, columns)

train_df, test_df = df_reg.randomSplit([0.7, 0.3], seed=42)
print("train:", train_df.count(), " test:", test_df.count())

assembler = VectorAssembler(inputCols=["year", "mileage", "engine_vol"], outputCol="features")
train_vec = assembler.transform(train_df)
test_vec = assembler.transform(test_df)

lr = LinearRegression(featuresCol="features", labelCol="price")
lr_model = lr.fit(train_vec)

print("Coefficients (year, mileage, engine_vol):", lr_model.coefficients)
print("Intercept:", lr_model.intercept)

predictions = lr_model.transform(test_vec)
predictions.select("id", "year", "mileage", "engine_vol", "price", "prediction").show()

r2 = RegressionEvaluator(labelCol="price", predictionCol="prediction", metricName="r2").evaluate(predictions)
print(f"R2 на тестовой выборке: {r2:.4f}")

train: 5  test: 3


Coefficients (year, mileage, engine_vol): [10448.629646190704,0.647380377609033,8366.898825933004]
Intercept: -21105867.257790577


+---+----+-------+----------+-----+------------------+
| id|year|mileage|engine_vol|price|        prediction|
+---+----+-------+----------+-----+------------------+
|  1|2010| 150000|       1.6| 5000| 6372.425815586001|
|  2|2015|  80000|       2.0|12000|16645.707144279033|
|  8|2019|  30000|       2.5|25000|30254.656261559576|
+---+----+-------+----------+-----+------------------+



R2 на тестовой выборке: 0.7521


## Задание №3: Одобрение кредита (Pipeline + LogisticRegression)

In [4]:
data = [
    (1, 50000, 700, "Employed", 1),
    (2, 20000, 600, "Unemployed", 0),
    (3, 80000, 750, "Employed", 1),
    (4, 30000, 620, "Employed", 0),
    (5, 120000, 800, "Employed", 1),
    (6, 15000, 550, "Unemployed", 0),
    (7, 45000, 680, "Employed", 1),
    (8, 25000, 590, "Unemployed", 0),
    (9, 60000, 720, "Employed", 1),
    (10, 35000, 640, "Employed", 0)
]
columns = ["client_id", "income", "credit_score", "status", "approved"]
df_credit = spark.createDataFrame(data, columns)

train_df, test_df = df_credit.randomSplit([0.7, 0.3], seed=42)
print("train:", train_df.count(), " test:", test_df.count())

status_indexer = StringIndexer(inputCol="status", outputCol="status_idx")
assembler = VectorAssembler(inputCols=["income", "credit_score", "status_idx"], outputCol="features")
log_reg = LogisticRegression(featuresCol="features", labelCol="approved", maxIter=10)

pipeline = Pipeline(stages=[status_indexer, assembler, log_reg])
credit_model = pipeline.fit(train_df)

pred_credit = credit_model.transform(test_df)
pred_credit.select("client_id", "income", "credit_score", "status", "approved",
                   "rawPrediction", "probability", "prediction").show(truncate=False)

auc = BinaryClassificationEvaluator(labelCol="approved", rawPredictionCol="rawPrediction",
                                    metricName="areaUnderROC").evaluate(pred_credit)
print(f"ROC AUC: {auc:.4f}")

train: 6  test: 4


+---------+------+------------+----------+--------+----------------------------------------+------------------------------------------+----------+
|client_id|income|credit_score|status    |approved|rawPrediction                           |probability                               |prediction|
+---------+------+------------+----------+--------+----------------------------------------+------------------------------------------+----------+
|1        |50000 |700         |Employed  |1       |[-8.52205882152414,8.52205882152414]    |[1.9898963282412432E-4,0.9998010103671758]|1.0       |
|2        |20000 |600         |Unemployed|0       |[10.445604369305741,-10.445604369305741]|[0.9999709250460875,2.907495391246684E-5] |0.0       |
|3        |80000 |750         |Employed  |1       |[-20.114716230689723,20.114716230689723]|[1.8377639800873238E-9,0.9999999981622361]|1.0       |
|10       |35000 |640         |Employed  |0       |[1.983713746682625,-1.983713746682625]  |[0.879076493230423,0.12092

ROC AUC: 1.0000


## Задание №4: Автопоиск лучших параметров (CrossValidator + RandomForest)

In [5]:
data = [
    (1, 7.4, 0.70, 0.0, 9.4, 0),
    (2, 7.8, 0.88, 0.0, 9.8, 0),
    (3, 7.8, 0.76, 0.04, 9.8, 1),
    (4, 11.2, 0.28, 0.56, 9.8, 1),
    (5, 7.4, 0.70, 0.0, 9.4, 0),
    (6, 7.4, 0.66, 0.0, 9.4, 0),
    (7, 7.9, 0.60, 0.06, 9.4, 1),
    (8, 7.3, 0.65, 0.0, 10.0, 1),
    (9, 7.8, 0.58, 0.02, 9.5, 1),
    (10, 7.5, 0.50, 0.36, 10.5, 1),
    (11, 6.7, 0.58, 0.08, 9.2, 0),
    (12, 7.5, 0.50, 0.36, 10.5, 1)
]
columns = ["id", "acidity", "volatile_acidity", "citric_acid", "alcohol", "quality"]
df_wine = spark.createDataFrame(data, columns)

train_df, test_df = df_wine.randomSplit([0.7, 0.3], seed=42)
print("train:", train_df.count(), " test:", test_df.count())

feature_cols = [c for c in df_wine.columns if c not in ("id", "quality")]
assembler = VectorAssembler(inputCols=feature_cols, outputCol="features")
train_vec = assembler.transform(train_df)
test_vec = assembler.transform(test_df)

rf = RandomForestClassifier(featuresCol="features", labelCol="quality", seed=42)
param_grid = (ParamGridBuilder()
              .addGrid(rf.maxDepth, [2, 4])
              .addGrid(rf.numTrees, [5, 10])
              .build())
evaluator = BinaryClassificationEvaluator(labelCol="quality", metricName="areaUnderROC")

cv = CrossValidator(estimator=rf, estimatorParamMaps=param_grid, evaluator=evaluator,
                    numFolds=2, seed=42)
cv_model = cv.fit(train_vec)

print("Средний AUC по фолдам для каждой комбинации параметров:")
for params, metric in zip(param_grid, cv_model.avgMetrics):
    print({p.name: v for p, v in params.items()}, "->", round(metric, 4))

best = cv_model.bestModel
print("Лучшее numTrees:", best.getNumTrees)
print("Лучшее maxDepth:", best.getOrDefault("maxDepth"))

pred_wine = cv_model.transform(test_vec)
pred_wine.select("id", "quality", "probability", "prediction").show(truncate=False)
print(f"ROC AUC на тесте: {evaluator.evaluate(pred_wine):.4f}")

train: 8  test: 4


Средний AUC по фолдам для каждой комбинации параметров:
{'maxDepth': 2, 'numTrees': 5} -> 1.0
{'maxDepth': 2, 'numTrees': 10} -> 0.8333
{'maxDepth': 4, 'numTrees': 5} -> 1.0
{'maxDepth': 4, 'numTrees': 10} -> 0.8333
Лучшее numTrees: 5
Лучшее maxDepth: 2


+---+-------+-----------+----------+
|id |quality|probability|prediction|
+---+-------+-----------+----------+
|2  |0      |[0.8,0.2]  |0.0       |
|3  |1      |[0.6,0.4]  |0.0       |
|4  |1      |[0.0,1.0]  |1.0       |
|12 |1      |[0.0,1.0]  |1.0       |
+---+-------+-----------+----------+



ROC AUC на тесте: 1.0000


## Задание №5: Предсказание оттока абонентов (GBTClassifier + Pipeline)

In [6]:
data = [
    (1, 45, "Month-to-month", 75.5, 1),
    (2, 60, "One year", 105.2, 0),
    (3, 22, "Month-to-month", 55.0, 1),
    (4, 30, "Two year", 20.5, 0),
    (5, 55, "One year", 90.0, 0),
    (6, 40, "Month-to-month", 85.5, 1),
    (7, 28, "Two year", 25.0, 0),
    (8, 65, "Month-to-month", 100.5, 1),
    (9, 50, "One year", 60.5, 0),
    (10, 35, "Two year", 30.0, 0),
    (11, 42, "Month-to-month", 80.0, 1),
    (12, 58, "One year", 95.0, 0),
    (13, 25, "Month-to-month", 50.0, 1),
    (14, 32, "Two year", 22.5, 0),
    (15, 52, "One year", 88.0, 0),
    (16, 38, "Month-to-month", 82.0, 1),
    (17, 29, "Two year", 26.0, 0),
    (18, 62, "Month-to-month", 98.0, 1),
    (19, 48, "One year", 65.0, 0),
    (20, 36, "Two year", 32.0, 0)
]
columns = ["client_id", "age", "contract_type", "monthly_charges", "churn"]
df_churn = spark.createDataFrame(data, columns)

train_df, test_df = df_churn.randomSplit([0.7, 0.3], seed=42)
print("train:", train_df.count(), " test:", test_df.count())

contract_indexer = StringIndexer(inputCol="contract_type", outputCol="contract_idx")
assembler = VectorAssembler(inputCols=["age", "contract_idx", "monthly_charges"], outputCol="features")
gbt = GBTClassifier(featuresCol="features", labelCol="churn", maxIter=10, seed=42)

churn_pipeline = Pipeline(stages=[contract_indexer, assembler, gbt])
churn_model = churn_pipeline.fit(train_df)

pred_churn = churn_model.transform(test_df)
pred_churn.select("client_id", "probability", "prediction").show(truncate=False)

auc = BinaryClassificationEvaluator(labelCol="churn", rawPredictionCol="rawPrediction",
                                    metricName="areaUnderROC").evaluate(pred_churn)
print(f"ROC AUC: {auc:.4f}")

train: 14  test: 6


+---------+----------------------------------------+----------+
|client_id|probability                             |prediction|
+---------+----------------------------------------+----------+
|2        |[0.9341221756527824,0.06587782434721756]|0.0       |
|4        |[0.9341221756527824,0.06587782434721756]|0.0       |
|5        |[0.9341221756527824,0.06587782434721756]|0.0       |
|6        |[0.06587782434721744,0.9341221756527825]|1.0       |
|10       |[0.9341221756527824,0.06587782434721756]|0.0       |
|19       |[0.9341221756527824,0.06587782434721756]|0.0       |
+---------+----------------------------------------+----------+



ROC AUC: 1.0000


In [7]:
spark.stop()